# Hello, Naive Bayes — classify by counting, not gradient descent

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sunilmogadati/production-ai-engineering/blob/main/notebooks/hello_naive_bayes.ipynb)

*Run each cell top to bottom (▶). Self-contained — no data files needed.*

Pairs with **ML_Study_05**. Logistic regression *learned* weights by rolling downhill. Naive Bayes does
something completely different: it just **counts** how often each clue went with each label, then
multiplies those fractions. No learning rate, no iterations — one pass over the data.

## 1. The dataset — 14 days of "should we play tennis?"
`PlayTennis` is the label (Yes/No); we use two clues, **Outlook** and **Temperature**.

In [ ]:
from collections import Counter

#            (Outlook,    Temperature, PlayTennis)
DATA = [
    ("Sunny",    "Hot",  "No"),  ("Sunny",    "Hot",  "No"),  ("Overcast", "Hot",  "Yes"),
    ("Rain",     "Mild", "Yes"), ("Rain",     "Cool", "Yes"), ("Rain",     "Cool", "No"),
    ("Overcast", "Cool", "Yes"), ("Sunny",    "Mild", "No"),  ("Sunny",    "Cool", "Yes"),
    ("Rain",     "Mild", "Yes"), ("Sunny",    "Mild", "Yes"), ("Overcast", "Mild", "Yes"),
    ("Overcast", "Hot",  "Yes"), ("Rain",     "Mild", "No"),
]
LABELS = ["Yes", "No"]
print(f"{len(DATA)} days")

## 2. Fit = count
For Naive Bayes, "fitting" is just tallying frequencies. Build the priors and per-feature tables.

In [ ]:
label_counts = Counter(row[2] for row in DATA)               # {Yes: 9, No: 5}
outlook_counts = {lab: Counter() for lab in LABELS}
temp_counts    = {lab: Counter() for lab in LABELS}
for outlook, temp, label in DATA:
    outlook_counts[label][outlook] += 1
    temp_counts[label][temp] += 1

total = sum(label_counts.values())
print(f"Priors:  P(Yes)={label_counts['Yes']}/{total}   P(No)={label_counts['No']}/{total}\n")
print("Outlook:")
for v in ["Sunny","Overcast","Rain"]:
    y,n = outlook_counts['Yes'][v], outlook_counts['No'][v]
    print(f"  {v:9} Yes={y} No={n}   P(.|Yes)={y/9:.3f}  P(.|No)={n/5:.3f}")
print("Temperature:")
for v in ["Hot","Mild","Cool"]:
    y,n = temp_counts['Yes'][v], temp_counts['No'][v]
    print(f"  {v:9} Yes={y} No={n}   P(.|Yes)={y/9:.3f}  P(.|No)={n/5:.3f}")

## 3. The scoring function
For each label: `prior × P(outlook|label) × P(temp|label)` (denominator dropped — it's the same for both).
`alpha=1` turns on **Laplace smoothing** so no count is ever exactly zero.

In [ ]:
n_out  = len({o for o,_,_ in DATA})   # 3 outlook categories
n_temp = len({t for _,t,_ in DATA})   # 3 temperature categories

def like(count, tot, alpha, k):
    return (count + alpha) / (tot + alpha*k)

def score(outlook, temp, label, alpha=0.0):
    prior  = label_counts[label] / total
    p_out  = like(outlook_counts[label][outlook], label_counts[label], alpha, n_out)
    p_temp = like(temp_counts[label][temp],       label_counts[label], alpha, n_temp)
    return prior * p_out * p_temp

def predict(outlook, temp, alpha=0.0):
    raw = {lab: score(outlook, temp, lab, alpha) for lab in LABELS}
    s = sum(raw.values())
    probs = {lab: (raw[lab]/s if s>0 else 0.0) for lab in LABELS}
    return max(probs, key=probs.get), raw, probs

## 4. Predict (Sunny, Hot)  — matches Study_05 Part 6–7

In [ ]:
winner, raw, probs = predict("Sunny", "Hot")
print(f"score(Yes) = {raw['Yes']:.3f}   (9/14 * 2/9 * 2/9)")
print(f"score(No)  = {raw['No']:.3f}   (5/14 * 3/5 * 2/5)")
print(f"normalized -> Yes: {probs['Yes']*100:.0f}%   No: {probs['No']*100:.0f}%")
print(f"PREDICTION: {winner}")

## 5. The assignment (Overcast, Mild) — and the zero-frequency trap
Overcast **never** occurred with "No" in the data → `P(Overcast|No) = 0/5`, and one zero in a product
kills the whole label. Watch the No score collapse to 0, then Laplace smoothing rescue it.

In [ ]:
winner, raw, probs = predict("Overcast", "Mild")             # no smoothing
print(f"No smoothing:  score(Yes)={raw['Yes']:.3f}  score(No)={raw['No']:.3f}  <- exactly 0")
print(f"               PREDICTION: {winner}\n")

winner_s, raw_s, probs_s = predict("Overcast", "Mild", alpha=1.0)   # Laplace
print(f"With Laplace:  P(Overcast|No) = (0+1)/(5+3) = {1/8:.3f}, not 0")
print(f"               Yes: {probs_s['Yes']*100:.0f}%   No: {probs_s['No']*100:.0f}%   PREDICTION: {winner_s}")

## 6. Confirm with scikit-learn
`CategoricalNB` does the same counting (with smoothing on by default) and agrees on the winners.

In [ ]:
from sklearn.naive_bayes import CategoricalNB
from sklearn.preprocessing import OrdinalEncoder

enc = OrdinalEncoder()
X = enc.fit_transform([[o, t] for o, t, _ in DATA])
y = [lab for _, _, lab in DATA]
clf = CategoricalNB(alpha=1.0).fit(X, y)

for clue in [["Sunny","Hot"], ["Overcast","Mild"]]:
    p = dict(zip(clf.classes_, clf.predict_proba(enc.transform([clue]))[0]))
    pretty = "  ".join(f"{k}:{v*100:.0f}%" for k,v in p.items())
    print(f"{tuple(clue)!s:22} -> {clf.predict(enc.transform([clue]))[0]:4} ({pretty})")

## Takeaway
- Naive Bayes **trains by counting** once — no iterations, no learning rate.
- It picks the label that makes your clues most probable (**Bayes' theorem**).
- **"Naive"** = it pretends the clues are independent, so it just multiplies them.
- Watch the **zero-frequency trap**; **Laplace smoothing** is the standard fix (sklearn does it by default).
- It's the classic fast baseline for **text / spam** — try it before anything fancier.